In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlkeneToAlcohol(MorphingOperator):
    def __init__(self):
        super(AlkeneToAlcohol, self).__init__()
        self._name = "Markovnikov Hydration"
        self._target_bonds = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")

    def setOriginal(self, mol):
        super(AlkeneToAlcohol, self).setOriginal(mol)
        self._target_bonds = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            bond = rdkit_mol.GetBondBetweenAtoms(match[0], match[1])
            if bond and not bond.GetIsAromatic():
                self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_bonds: return MolpherMol(other=rdkit_mol)
            
        idx1, idx2 = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            h1 = rw_mol.GetAtomWithIdx(idx1).GetTotalNumHs()
            h2 = rw_mol.GetAtomWithIdx(idx2).GetTotalNumHs()
            idx_with_oh = idx1 if h1 <= h2 else idx2
            
            oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(idx_with_oh, oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            for idx in [idx1, idx2, oh_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name

op_hydration = AlkeneToAlcohol()
hydration_tests = {
    "1. 2-μεθυλο-2-βουτένιο (Αλκένιο -> Αλκοόλη)": "CC=C(C)C",
    "2. Βενζόλιο (Αρωματικό -> Πρέπει να αγνοηθεί)": "c1ccccc1"
}

print("=== STARTING MARKOVNIKOV HYDRATION TESTING ===")
for name, smiles in hydration_tests.items():
    mol = MolpherMol(smiles)
    op_hydration.setOriginal(mol)
    product = op_hydration.morph()
    print(f"\n{name}\n  SOURCE: {mol.getSMILES()}\n  TARGET: {product.getSMILES() if product and product.getSMILES() != mol.getSMILES() else 'No change (Safe)'}")



=== STARTING MARKOVNIKOV HYDRATION TESTING ===

1. 2-μεθυλο-2-βουτένιο (Αλκένιο -> Αλκοόλη)
  SOURCE: CC=C(C)C
  TARGET: CCC(C)(C)O

2. Βενζόλιο (Αρωματικό -> Πρέπει να αγνοηθεί)
  SOURCE: C1=CC=CC=C1
  TARGET: No change (Safe)


In [2]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AlkeneToAlcohol(MorphingOperator):
    def __init__(self):
        super(AlkeneToAlcohol, self).__init__()
        self._name = "Markovnikov Hydration"
        self._target_bonds = [] 
        self.PATTERN = Chem.MolFromSmarts("[CX3;H1,H2]=[CX3;H0,H1,H2]")

    def setOriginal(self, mol):
        super(AlkeneToAlcohol, self).setOriginal(mol)
        self._target_bonds = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            bond = rdkit_mol.GetBondBetweenAtoms(match[0], match[1])
            if bond and not bond.GetIsAromatic():
                self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        if not self._target_bonds: return MolpherMol(other=rdkit_mol)
            
        idx1, idx2 = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx1, idx2)
            if bond:
                bond.SetBondType(Chem.BondType.SINGLE)
            
            h1 = rw_mol.GetAtomWithIdx(idx1).GetTotalNumHs()
            h2 = rw_mol.GetAtomWithIdx(idx2).GetTotalNumHs()
            idx_with_oh = idx1 if h1 <= h2 else idx2
            
            oh_idx = rw_mol.AddAtom(Chem.Atom(8))
            rw_mol.AddBond(idx_with_oh, oh_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            for idx in [idx1, idx2, oh_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)
    
    def getName(self): return self._name

op_hydration = AlkeneToAlcohol()
start_mol = MolpherMol("CC=C(C)C")          
target_mol = MolpherMol("CCC(C)(C)O")   
tree = ETree.create(source=start_mol, target=target_mol)    
tree.morphing_operators = (op_hydration,)

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target

closest_info = FindClosest()
max_generations = 5

print("=== STARTING MARKOVNIKOV HYDRATION PATHWAY SEARCH ===")
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print('Generation #', tree.generation_count, sep='')
    print('Molecules in tree:', tree.mol_count)
    if closest_info.closest_mol:
        print('Closest molecule to target: {0} (Tanimoto distance: {1})'.format(
            closest_info.closest_mol.getSMILES(), closest_info.closest_distance))
    print("-" * 40)

if tree.path_found:
    print("SUCCESS!")
else:
    print("FAILED: Δεν βρέθηκε το μονοπάτι.")
print("=========================================")

=== STARTING MARKOVNIKOV HYDRATION PATHWAY SEARCH ===
Generation #1
Molecules in tree: 2
Closest molecule to target: CCC(C)(C)O (Tanimoto distance: 0.0)
----------------------------------------
SUCCESS!
